In [2]:
from google.colab import files

uploaded = files.upload()

Saving gprofiler_down.csv to gprofiler_down.csv
Saving gprofiler_up.csv to gprofiler_up.csv
Saving expression_log2_normalized.csv to expression_log2_normalized.csv
Saving protein_annotation.csv to protein_annotation.csv
Saving sample_annotation.csv to sample_annotation.csv


In [5]:
# ============================================================
# PR1 PROTEOMICS
# FINAL FIVE PATHWAY-BASED HEATMAPS
#
# Comparison: AS vs CON
#
# Four groups:
# CON-F | CON-M | AS-F | AS-M
#
# Uses:
# - Observed log2-normalized expression values
# - gProfiler UP and DOWN results
# - Protein annotation
# - Peptide-support filtering
# - Minimum 50% valid observations in every subgroup
#
# Outputs:
# - 5 PNG heatmaps
# - Optional PDF heatmaps
# - Protein membership audit CSVs
# - Enrichment term audit CSVs
# - Gene selection audit CSVs
# - Category overlap CSV
# - Summary CSV
# - One ZIP package
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from itertools import combinations
from IPython.display import display


sns.set_theme(
    style="white",
    font="DejaVu Sans"
)


# ------------------------------------------------------------
# 2. USER SETTINGS
# ------------------------------------------------------------

EXPRESSION_FILE = "expression_log2_normalized.csv"
PROTEIN_ANNOTATION_FILE = "protein_annotation.csv"
SAMPLE_ANNOTATION_FILE = "sample_annotation.csv"

GPROFILER_UP_FILE = "gprofiler_up.csv"
GPROFILER_DOWN_FILE = "gprofiler_down.csv"

OUTPUT_DIR = "PR1_final_heatmaps"

# Minimum fraction of samples with observed values
# required in EACH of the four groups.

MIN_VALID_FRAC = 0.50

# Keep False to create exactly one PNG per category.
# Change to True if you also need vector PDF files.

SAVE_PDF = False

# Fixed order of the four groups

GROUP_ORDER = [
    "CON_F",
    "CON_M",
    "AS_F",
    "AS_M"
]

DISPLAY_GROUPS = {
    "CON_F": "CON-F",
    "CON_M": "CON-M",
    "AS_F": "AS-F",
    "AS_M": "AS-M"
}

# The gProfiler p_value column is used as provided.
# Only terms with p_value <= 0.05 are retained.

TERM_P_CUTOFF = 0.05


# ------------------------------------------------------------
# 3. CHECK REQUIRED FILES
# ------------------------------------------------------------

required_files = [
    EXPRESSION_FILE,
    PROTEIN_ANNOTATION_FILE,
    SAMPLE_ANNOTATION_FILE,
    GPROFILER_UP_FILE,
    GPROFILER_DOWN_FILE
]

missing_files = [
    f for f in required_files
    if not Path(f).exists()
]

if missing_files:

    raise FileNotFoundError(
        "The following files are missing:\n"
        + "\n".join(missing_files)
        + "\n\nPlease upload them using Cell 1."
    )


# Remove old output folder so that previous results
# cannot accidentally be included in the final ZIP.

if os.path.exists(OUTPUT_DIR):

    shutil.rmtree(OUTPUT_DIR)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

expression_raw = pd.read_csv(
    EXPRESSION_FILE
)

protein_annotation = pd.read_csv(
    PROTEIN_ANNOTATION_FILE
)

sample_annotation = pd.read_csv(
    SAMPLE_ANNOTATION_FILE
)

gprofiler_up = pd.read_csv(
    GPROFILER_UP_FILE
)

gprofiler_down = pd.read_csv(
    GPROFILER_DOWN_FILE
)


print("INPUT FILES LOADED")
print("------------------------------")

print("Expression matrix:", expression_raw.shape)
print("Protein annotation:", protein_annotation.shape)
print("Sample annotation:", sample_annotation.shape)
print("gProfiler UP:", gprofiler_up.shape)
print("gProfiler DOWN:", gprofiler_down.shape)


# ------------------------------------------------------------
# 5. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

def check_columns(df, required, filename):

    missing = [
        col for col in required
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"Missing columns in {filename}: {missing}"
        )


check_columns(
    expression_raw,
    ["protein_group_id"],
    EXPRESSION_FILE
)

check_columns(
    protein_annotation,
    [
        "protein_group_id",
        "gene_name",
        "majority_protein_ids",
        "razor_unique_peptides",
        "low_peptide_support"
    ],
    PROTEIN_ANNOTATION_FILE
)

check_columns(
    sample_annotation,
    [
        "sample",
        "group",
        "sex"
    ],
    SAMPLE_ANNOTATION_FILE
)

check_columns(
    gprofiler_up,
    [
        "source",
        "native",
        "name",
        "p_value",
        "intersection_size",
        "genes"
    ],
    GPROFILER_UP_FILE
)

check_columns(
    gprofiler_down,
    [
        "source",
        "native",
        "name",
        "p_value",
        "intersection_size",
        "genes"
    ],
    GPROFILER_DOWN_FILE
)


# ------------------------------------------------------------
# 6. PREPARE EXPRESSION MATRIX
# ------------------------------------------------------------

expression_raw["protein_group_id"] = (
    expression_raw["protein_group_id"]
    .astype(str)
    .str.strip()
)

if expression_raw["protein_group_id"].duplicated().any():

    raise ValueError(
        "Duplicate protein_group_id values in expression matrix."
    )


expression = (
    expression_raw
    .set_index("protein_group_id")
    .copy()
)

expression = expression.apply(
    pd.to_numeric,
    errors="coerce"
)


print("\nExpression matrix dimensions:", expression.shape)


# ------------------------------------------------------------
# 7. PREPARE SAMPLE ANNOTATION
# ------------------------------------------------------------

sample_annotation["sample"] = (
    sample_annotation["sample"]
    .astype(str)
    .str.strip()
)

if sample_annotation["sample"].duplicated().any():

    raise ValueError(
        "Duplicate sample IDs found in sample annotation."
    )


# IMPORTANT:
# Use the actual group metadata instead of inferring
# condition from the sample ID.

condition_map = {
    "control (donor heart)": "CON",
    "aortic stenosis": "AS",
    "mitral regurgitation": "MR"
}

sex_map = {
    "female": "F",
    "male": "M"
}


sample_annotation["condition_code"] = (
    sample_annotation["group"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .map(condition_map)
)

sample_annotation["sex_code"] = (
    sample_annotation["sex"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .map(sex_map)
)


if sample_annotation["condition_code"].isna().any():

    raise ValueError(
        "Some sample groups could not be mapped."
    )

if sample_annotation["sex_code"].isna().any():

    raise ValueError(
        "Some sample sexes could not be mapped."
    )


sample_annotation["plot_group"] = (
    sample_annotation["condition_code"]
    + "_"
    + sample_annotation["sex_code"]
)


# Check that expression and annotation contain
# exactly the same sample IDs.

expression_samples = set(expression.columns)

annotation_samples = set(
    sample_annotation["sample"]
)

if expression_samples != annotation_samples:

    raise ValueError(
        "Sample IDs do not match between expression "
        "and sample annotation.\n"
        f"Missing in expression: "
        f"{sorted(annotation_samples - expression_samples)}\n"
        f"Missing in annotation: "
        f"{sorted(expression_samples - annotation_samples)}"
    )


# ------------------------------------------------------------
# 8. DEFINE THE FOUR GROUPS
# ------------------------------------------------------------

group_samples = {}

for group in GROUP_ORDER:

    samples = sample_annotation.loc[
        sample_annotation["plot_group"] == group,
        "sample"
    ].tolist()

    if len(samples) == 0:

        raise ValueError(
            f"No samples found for {group}."
        )

    group_samples[group] = samples


group_count_table = pd.DataFrame({

    "plot_group": GROUP_ORDER,

    "display_group": [
        DISPLAY_GROUPS[g]
        for g in GROUP_ORDER
    ],

    "n_samples": [
        len(group_samples[g])
        for g in GROUP_ORDER
    ]

})


print("\nSAMPLE COUNTS")
print("------------------------------")

display(group_count_table)


group_count_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sample_group_counts.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 9. CALCULATE GROUP MEANS AND COVERAGE
# ------------------------------------------------------------

# Use observed normalized values.
#
# Do NOT use the imputed expression file here.
#
# For each protein:
# - calculate the mean log2 intensity per subgroup
# - calculate the fraction of observed values per subgroup
#
# The mean is calculated using available observations.
# Missing values remain missing.


group_means = pd.DataFrame(
    index=expression.index
)

group_coverage = pd.DataFrame(
    index=expression.index
)


for group in GROUP_ORDER:

    samples = group_samples[group]

    group_means[group] = (
        expression[samples]
        .mean(axis=1, skipna=True)
    )

    group_coverage[group] = (
        expression[samples]
        .notna()
        .sum(axis=1)
        / len(samples)
    )


# ------------------------------------------------------------
# 10. PREPARE PROTEIN ANNOTATION
# ------------------------------------------------------------

protein_annotation["protein_group_id"] = (
    protein_annotation["protein_group_id"]
    .astype(str)
    .str.strip()
)


if protein_annotation["protein_group_id"].duplicated().any():

    raise ValueError(
        "Duplicate protein_group_id values in protein annotation."
    )


def parse_boolean(value):

    """
    Convert low_peptide_support values to Boolean.

    Missing or unknown values are conservatively excluded.
    """

    if pd.isna(value):

        return True

    if isinstance(value, (bool, np.bool_)):

        return bool(value)

    value = str(value).strip().lower()

    if value in ["true", "1", "yes", "y"]:

        return True

    if value in ["false", "0", "no", "n"]:

        return False

    if value in ["", "na", "nan", "none"]:

        return True

    raise ValueError(
        f"Unrecognized Boolean value: {value}"
    )


protein_annotation["low_peptide_support_bool"] = (
    protein_annotation["low_peptide_support"]
    .apply(parse_boolean)
)


# Save all excluded low-support groups for transparency.

excluded_low_support = protein_annotation.loc[
    protein_annotation["low_peptide_support_bool"]
].copy()


excluded_low_support.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "excluded_low_peptide_support_proteins.csv"
    ),
    index=False
)


# Keep only protein groups with sufficient peptide support.

protein_annotation_good = (
    protein_annotation.loc[
        ~protein_annotation["low_peptide_support_bool"]
    ].copy()
)


# Keep only IDs that exist in expression matrix.

protein_annotation_good = (
    protein_annotation_good.loc[
        protein_annotation_good["protein_group_id"]
        .isin(expression.index)
    ].copy()
)


print("\nPEPTIDE SUPPORT FILTER")
print("------------------------------")

print(
    "Excluded low-support protein groups:",
    len(excluded_low_support)
)

print(
    "Protein groups retained:",
    len(protein_annotation_good)
)


# ------------------------------------------------------------
# 11. HELPER FUNCTIONS FOR GENE SYMBOLS
# ------------------------------------------------------------

def parse_annotation_symbols(value):

    """
    Split gene annotations such as:
        COL1A1
        TGM2;TGM5
        PLG;PLGLB1

    Returns uppercase gene symbols.
    """

    if pd.isna(value):

        return set()

    parts = re.split(
        r"[;,|]",
        str(value)
    )

    return {
        part.strip().upper()
        for part in parts
        if part.strip()
        and part.strip().lower() not in [
            "nan", "na", "none"
        ]
    }


def parse_gprofiler_genes(value):

    """
    Split comma-separated or semicolon-separated
    gProfiler gene lists.
    """

    if pd.isna(value):

        return set()

    parts = re.split(
        r"[,;]",
        str(value)
    )

    return {
        part.strip().upper()
        for part in parts
        if part.strip()
    }


protein_annotation_good["gene_symbols"] = (
    protein_annotation_good["gene_name"]
    .apply(parse_annotation_symbols)
)


# Union of all gene symbols represented by
# sufficiently supported protein groups.

annotated_gene_symbols = set()

for symbols in protein_annotation_good["gene_symbols"]:

    annotated_gene_symbols.update(symbols)


# ------------------------------------------------------------
# 12. DEFINE MANUAL CURATION RULES
# ------------------------------------------------------------

# These are transparent gene-level exclusions.
#
# They do NOT alter the source gProfiler results.
#
# They only decide which heatmap receives a gene.
#
# The exclusions are exported in gene_audit.csv files.


ECM_EXCLUSIONS = {

    "FGA": "Fibrinogen / coagulation; assigned to category 5",
    "FGB": "Fibrinogen / coagulation; assigned to category 5",
    "FGG": "Fibrinogen / coagulation; assigned to category 5",
    "PLG": "Fibrinolysis; assigned to category 5",
    "SERPINF2": "Fibrinolysis; assigned to category 5",
    "C1QC": "Complement; assigned to category 5",
    "ACTN1": "Cytoskeletal / adhesion protein; assigned to category 4",
    "PLEC": "Cytoskeletal linker; assigned to category 4"

}


CYTOSKELETON_EXCLUSIONS = {

    "AMBP": "Plasma-associated protein; category 5",
    "APOA1": "Lipoprotein / plasma-associated; category 5",
    "APOA4": "Lipoprotein / plasma-associated; category 5",
    "AZGP1": "Plasma-associated protein; category 5",
    "C3": "Complement protein; category 5",
    "FGA": "Fibrinogen / coagulation; category 5",
    "FGB": "Fibrinogen / coagulation; category 5",
    "FGG": "Fibrinogen / coagulation; category 5",
    "HBB": "Blood-associated protein; category 5",
    "HRG": "Plasma / coagulation-associated protein; category 5",
    "IGFALS": "Circulating plasma protein; category 5",
    "KNG1": "Coagulation-associated protein; category 5",
    "PLG": "Fibrinolysis; category 5",
    "SERPINE2": "Coagulation / protease-inhibitor-associated; category 5",
    "SERPINF2": "Fibrinolysis; category 5",
    "TTR": "Circulating transport protein; not a core cytoskeletal protein"

}


# ------------------------------------------------------------
# 13. DEFINE THE FIVE FINAL CATEGORIES
# ------------------------------------------------------------

CATEGORY_SPECS = [

    # --------------------------------------------------------
    # CATEGORY 1
    # ECM COMPOSITION AND REMODELING
    # --------------------------------------------------------

    {
        "slug": "01_ECM",

        "title": "ECM composition and remodeling",

        "source": "up",

        "terms": [

            "extracellular matrix",
            "external encapsulating structure",
            "Extracellular matrix organization",
            "collagen trimer",
            "Collagen chain trimerization",
            "ECM proteoglycans",
            "Degradation of the extracellular matrix",
            "collagen fibril organization"

        ],

        "exclusions": ECM_EXCLUSIONS,

        "context_terms": []
    },


    # --------------------------------------------------------
    # CATEGORY 2
    # ENERGY METABOLISM AND MITOCHONDRIA
    # --------------------------------------------------------

    {
        "slug": "02_Mitochondria",

        "title": "Energy metabolism and mitochondria",

        "source": "down",

        "terms": [

            "mitochondrion",
            "mitochondrial matrix",
            "mitochondrial gene expression"

        ],

        "exclusions": {},

        "context_terms": []
    },


    # --------------------------------------------------------
    # CATEGORY 3
    # PROTEIN SYNTHESIS AND RIBOSOME
    # --------------------------------------------------------

    {
        "slug": "03_Protein_synthesis",

        "title": "Protein synthesis and ribosomal machinery",

        "source": "down",

        "terms": [

            "Translation",
            "Ribosome",
            "ribosomal subunit",
            "ribosome",
            "translation",
            "protein biosynthetic process",
            "SRP-dependent cotranslational protein targeting to membrane"

        ],

        "exclusions": {},

        "context_terms": []
    },


    # --------------------------------------------------------
    # CATEGORY 4
    # CYTOSKELETON, ADHESION AND MUSCLE CONTRACTION
    # --------------------------------------------------------

    {
        "slug": "04_Cytoskeleton",

        "title": "Cytoskeleton, adhesion and muscle contraction",

        "source": "up",

        "terms": [

            "Cytoskeleton in muscle cells",
            "cell adhesion",
            "Integrin cell surface interactions",
            "Integrin signaling",
            "Non-integrin membrane-ECM interactions"

        ],

        "exclusions": CYTOSKELETON_EXCLUSIONS,

        "context_terms": []
    },


    # --------------------------------------------------------
    # CATEGORY 5
    # BLOOD / PLASMA, COMPLEMENT AND COAGULATION
    # --------------------------------------------------------

    {
        "slug": "05_Blood_plasma_complement_coagulation",

        "title": "Blood/plasma-associated proteins, complement and coagulation",

        "source": "up",

        "terms": [

            "Complement and coagulation cascades",
            "Complement cascade",
            "Regulation of Complement cascade",

            "complement activation",
            "complement activation, classical pathway",
            "complement activation, alternative pathway",
            "Initial triggering of complement",

            "humoral immune response",
            "humoral immune response mediated by circulating immunoglobulin",

            "regulation of coagulation",
            "negative regulation of coagulation",
            "regulation of blood coagulation",
            "negative regulation of blood coagulation",

            "coagulation",
            "hemostasis",
            "blood coagulation",

            "fibrinolysis",
            "regulation of fibrinolysis",
            "negative regulation of fibrinolysis",

            "Formation of Fibrin Clot (Clotting Cascade)",
            "Common Pathway of Fibrin Clot Formation",
            "Intrinsic Pathway of Fibrin Clot Formation",

            "Platelet degranulation",
            "platelet alpha granule",
            "platelet alpha granule lumen",
            "Platelet activation, signaling and aggregation",
            "Response to elevated platelet cytosolic Ca2+",

            "protein activation cascade"

        ],

        "exclusions": {},

        # This term was a major reason for defining
        # category 5, but it is broad.
        # It is reported as context, not used to
        # directly add all 52 genes to this heatmap.

        "context_terms": [
            "blood microparticle"
        ]
    }

]


# ------------------------------------------------------------
# 14. HELPER FUNCTION: EXTRACT ENRICHED TERMS AND GENES
# ------------------------------------------------------------

def extract_category_data(category_spec):

    if category_spec["source"] == "up":

        source_df = gprofiler_up.copy()

    elif category_spec["source"] == "down":

        source_df = gprofiler_down.copy()

    else:

        raise ValueError("Unknown enrichment source.")


    source_df["_term_norm"] = (
        source_df["name"]
        .astype(str)
        .str.strip()
        .str.casefold()
    )


    requested_terms = {
        term.strip().casefold()
        for term in category_spec["terms"]
    }


    found_terms = set(source_df["_term_norm"])


    missing_terms = sorted(
        requested_terms - found_terms
    )


    all_matching_rows = source_df.loc[
        source_df["_term_norm"].isin(requested_terms)
    ].copy()


    all_matching_rows["p_value"] = pd.to_numeric(
        all_matching_rows["p_value"],
        errors="coerce"
    )


    non_significant_terms = (
        all_matching_rows.loc[
            all_matching_rows["p_value"] > TERM_P_CUTOFF,
            "name"
        ]
        .astype(str)
        .tolist()
    )


    # Retain enriched terms passing the configured p-value cutoff.

    selected_terms = all_matching_rows.loc[
        all_matching_rows["p_value"] <= TERM_P_CUTOFF
    ].copy()


    if len(selected_terms) == 0:

        raise ValueError(
            f"No significant terms found for {category_spec['title']}."
        )


    # Extract the union of gene symbols.

    selected_genes = set()

    for gene_list in selected_terms["genes"]:

        selected_genes.update(
            parse_gprofiler_genes(gene_list)
        )


    # Remove internal helper column.

    selected_terms = selected_terms.drop(
        columns=["_term_norm"]
    )


    return (
        selected_genes,
        selected_terms,
        missing_terms,
        non_significant_terms,
        source_df
    )


# ------------------------------------------------------------
# 15. FUNCTION TO FIND CONTEXT-ONLY TERMS
# ------------------------------------------------------------

def extract_context_terms(category_spec, source_df):

    context_names = {
        term.strip().casefold()
        for term in category_spec["context_terms"]
    }


    if not context_names:

        return pd.DataFrame(
            columns=source_df.columns.drop(
                labels=["_term_norm"],
                errors="ignore"
            )
        )


    context_df = source_df.loc[
        source_df["_term_norm"].isin(context_names)
    ].copy()


    context_df["p_value"] = pd.to_numeric(
        context_df["p_value"],
        errors="coerce"
    )


    context_df = context_df.drop(
        columns=["_term_norm"]
    )


    return context_df


# ------------------------------------------------------------
# 16. FUNCTION TO MAKE GENE-TO-TERM AUDIT
# ------------------------------------------------------------

def make_gene_term_map(selected_terms):

    gene_to_terms = {}

    for _, row in selected_terms.iterrows():

        term_name = str(row["name"])

        genes = parse_gprofiler_genes(
            row["genes"]
        )

        for gene in genes:

            gene_to_terms.setdefault(
                gene,
                []
            ).append(term_name)


    return gene_to_terms


# ------------------------------------------------------------
# 17. CREATE ONE HEATMAP
# ------------------------------------------------------------

def create_category_heatmap(category_spec):


    print("\n")
    print("=" * 75)
    print(category_spec["title"])
    print("=" * 75)


    # --------------------------------------------------------
    # A. EXTRACT ENRICHMENT GENES
    # --------------------------------------------------------

    (
        genes_before_curation,
        selected_terms,
        missing_terms,
        non_significant_terms,
        source_df
    ) = extract_category_data(category_spec)


    context_terms = extract_context_terms(
        category_spec,
        source_df
    )


    print(
        "Enriched term rows used:",
        len(selected_terms)
    )

    print(
        "Gene symbols before manual curation:",
        len(genes_before_curation)
    )


    if missing_terms:

        print(
            "WARNING - requested terms not found:",
            missing_terms
        )


    if non_significant_terms:

        print(
            "Terms excluded for p-value > cutoff:",
            non_significant_terms
        )


    # --------------------------------------------------------
    # B. APPLY MANUAL GENE CURATION
    # --------------------------------------------------------

    exclusions = category_spec["exclusions"]


    excluded_genes = (
        genes_before_curation
        .intersection(exclusions.keys())
    )


    genes_after_curation = (
        genes_before_curation - excluded_genes
    )


    print(
        "Genes excluded by manual curation:",
        sorted(excluded_genes)
    )

    print(
        "Gene symbols after curation:",
        len(genes_after_curation)
    )


    if len(genes_after_curation) == 0:

        raise ValueError(
            "No genes remain after manual curation."
        )


    # --------------------------------------------------------
    # C. MAP GENES TO PROTEIN GROUPS
    # --------------------------------------------------------

    annotation = protein_annotation_good.copy()


    annotation["matching_gene_symbols"] = (
        annotation["gene_symbols"].apply(
            lambda symbols: sorted(
                symbols.intersection(genes_after_curation)
            )
        )
    )


    matched_annotation = annotation.loc[
        annotation["matching_gene_symbols"].apply(len) > 0
    ].copy()


    matched_annotation["matching_gene_symbols_text"] = (
        matched_annotation["matching_gene_symbols"]
        .apply(lambda x: "; ".join(x))
    )


    matched_ids = (
        matched_annotation["protein_group_id"]
        .drop_duplicates()
        .tolist()
    )


    # Gene symbols that cannot be mapped to a
    # peptide-filtered protein group.

    mapped_gene_symbols = set()

    for symbols in matched_annotation["matching_gene_symbols"]:

        mapped_gene_symbols.update(symbols)


    unmatched_genes = (
        genes_after_curation - mapped_gene_symbols
    )


    print(
        "Matched protein groups:",
        len(matched_ids)
    )

    print(
        "Unmatched gene symbols:",
        sorted(unmatched_genes)
    )


    if len(matched_ids) == 0:

        raise ValueError(
            f"No protein groups mapped for {category_spec['title']}."
        )


    # --------------------------------------------------------
    # D. EXTRACT GROUP MEANS AND COVERAGE
    # --------------------------------------------------------

    means = group_means.loc[
        matched_ids,
        GROUP_ORDER
    ].copy()


    coverage = group_coverage.loc[
        matched_ids,
        GROUP_ORDER
    ].copy()


    # --------------------------------------------------------
    # E. APPLY COVERAGE FILTER
    # --------------------------------------------------------

    # A protein must have at least 50% observed values
    # in EACH of the four groups.

    coverage_pass_mask = (
        coverage >= MIN_VALID_FRAC
    ).all(axis=1)


    coverage_pass_ids = (
        coverage.index[coverage_pass_mask]
    )


    means_coverage_pass = means.loc[
        coverage_pass_ids
    ].copy()


    coverage_pass = coverage.loc[
        coverage_pass_ids
    ].copy()


    print(
        "Protein groups passing coverage filter:",
        len(coverage_pass_ids)
    )


    if len(means_coverage_pass) < 2:

        raise ValueError(
            "Too few protein groups passed the coverage filter."
        )


    # --------------------------------------------------------
    # F. ROW-WISE Z-SCORE
    # --------------------------------------------------------

    # Center and scale each protein across the four groups.
    #
    # Use sample standard deviation (ddof=1).
    #
    # This is consistent with row-wise scaling in the
    # pheatmap-style approach used in the paper.


    row_mean = means_coverage_pass.mean(
        axis=1
    )


    row_sd = means_coverage_pass.std(
        axis=1,
        ddof=1
    )


    # Remove proteins with no variation across the four groups.

    variable_mask = row_sd > 1e-8


    variable_ids = row_sd.index[
        variable_mask
    ]


    means_variable = means_coverage_pass.loc[
        variable_ids
    ].copy()


    row_mean_variable = row_mean.loc[
        variable_ids
    ]

    row_sd_variable = row_sd.loc[
        variable_ids
    ]


    z_scores = means_variable.sub(
        row_mean_variable,
        axis=0
    ).div(
        row_sd_variable,
        axis=0
    )


    if len(z_scores) < 2:

        raise ValueError(
            "Too few variable proteins remain for clustering."
        )


    print(
        "Protein groups included in heatmap:",
        len(z_scores)
    )


    # --------------------------------------------------------
    # G. GENERATE READABLE PROTEIN LABELS
    # --------------------------------------------------------

    annotation_lookup = (
        protein_annotation_good
        .set_index("protein_group_id")
    )


    raw_labels = {}

    for protein_id in z_scores.index:

        gene_name = annotation_lookup.loc[
            protein_id,
            "gene_name"
        ]


        if (
            pd.isna(gene_name)
            or str(gene_name).strip() == ""
        ):

            gene_name = protein_id


        raw_labels[protein_id] = str(gene_name)


    label_counts = pd.Series(
        list(raw_labels.values())
    ).value_counts()


    labels = {}

    for protein_id, gene_name in raw_labels.items():

        if label_counts[gene_name] > 1:

            labels[protein_id] = (
                gene_name
                + " ["
                + protein_id
                + "]"
            )

        else:

            labels[protein_id] = gene_name


    z_plot = z_scores.copy()


    z_plot.index = [
        labels[protein_id]
        for protein_id in z_scores.index
    ]


    z_plot.columns = [
        DISPLAY_GROUPS[group]
        for group in GROUP_ORDER
    ]


    # --------------------------------------------------------
    # H. DETERMINE FIGURE SIZE
    # --------------------------------------------------------

    n_proteins = len(z_plot)


    figure_height = max(
        9,
        min(
            32,
            5 + 0.15 * n_proteins
        )
    )


    if n_proteins > 120:

        ytick_fontsize = 5.5

    elif n_proteins > 80:

        ytick_fontsize = 6

    else:

        ytick_fontsize = 7


    # --------------------------------------------------------
    # I. CREATE CLUSTERED HEATMAP
    # --------------------------------------------------------

    cluster = sns.clustermap(

        z_plot,

        # Cluster protein rows.

        row_cluster=True,

        # Keep four biological groups in fixed order.

        col_cluster=False,

        # Color palette.

        cmap="vlag",

        center=0,

        vmin=-2,
        vmax=2,

        # Dimensions.

        figsize=(14, figure_height),

        # Clustering.
        #
        # pheatmap's default hierarchical clustering method
        # is complete linkage with Euclidean distance.

        method="complete",
        metric="euclidean",

        # Labels.

        xticklabels=True,
        yticklabels=True,

        # Appearance.

        linewidths=0,

        dendrogram_ratio=(0.12, 0.02),

        # Colorbar in upper-left area.

        cbar_pos=(0.02, 0.82, 0.025, 0.14),

        cbar_kws={
            "ticks": [-2, -1, 0, 1, 2]
        }

    )


    # --------------------------------------------------------
    # J. FORMAT HEATMAP
    # --------------------------------------------------------

    cluster.ax_heatmap.set_xlabel(
        "Condition and sex",
        fontsize=11,
        labelpad=8
    )

    cluster.ax_heatmap.set_ylabel("")


    cluster.ax_heatmap.tick_params(
        axis="x",
        labelsize=11,
        rotation=0
    )


    cluster.ax_heatmap.tick_params(
        axis="y",
        labelsize=ytick_fontsize,
        length=0,
        pad=2
    )


    # Move the colorbar label above the color scale.

    cluster.cax.set_ylabel("")

    cluster.cax.set_title(
        "Row Z-score",
        fontsize=8,
        pad=5
    )


    cluster.fig.suptitle(
        category_spec["title"],
        fontsize=15,
        fontweight="bold",
        y=1.02
    )


    # --------------------------------------------------------
    # K. SAVE PNG
    # --------------------------------------------------------

    slug = category_spec["slug"]


    png_path = os.path.join(
        OUTPUT_DIR,
        slug + "_heatmap.png"
    )


    cluster.fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight"
    )


    # Optional PDF.

    pdf_path = None

    if SAVE_PDF:

        pdf_path = os.path.join(
            OUTPUT_DIR,
            slug + "_heatmap.pdf"
        )

        cluster.fig.savefig(
            pdf_path,
            bbox_inches="tight"
        )


    plt.close(cluster.fig)


    # --------------------------------------------------------
    # L. SAVE PROTEIN MEMBERSHIP AUDIT
    # --------------------------------------------------------

    member_columns = [
        "protein_group_id",
        "gene_name",
        "majority_protein_ids",
        "razor_unique_peptides",
        "matching_gene_symbols_text"
    ]


    member_table = matched_annotation[
        member_columns
    ].copy()


    # Explicitly distinguish the two filters.

    member_table["passed_coverage_filter"] = (
        member_table["protein_group_id"]
        .isin(coverage_pass_ids)
    )


    member_table["had_variation_across_groups"] = (
        member_table["protein_group_id"]
        .isin(variable_ids)
    )


    member_table["included_in_heatmap"] = (
        member_table["protein_group_id"]
        .isin(z_scores.index)
    )


    for group in GROUP_ORDER:

        display_name = DISPLAY_GROUPS[group]


        member_table[
            f"coverage_{display_name}"
        ] = (
            member_table["protein_group_id"]
            .map(coverage[group])
        )


        member_table[
            f"mean_log2_{display_name}"
        ] = (
            member_table["protein_group_id"]
            .map(means[group])
        )


        member_table[
            f"zscore_{display_name}"
        ] = (
            member_table["protein_group_id"]
            .map(z_scores[group])
        )


    members_path = os.path.join(
        OUTPUT_DIR,
        slug + "_protein_members.csv"
    )


    member_table.to_csv(
        members_path,
        index=False
    )


    # --------------------------------------------------------
    # M. SAVE ENRICHMENT TERM AUDIT
    # --------------------------------------------------------

    terms_output = selected_terms.copy()

    terms_output["term_role"] = "gene_membership"

    terms_output["used_for_gene_selection"] = True

    terms_output["curation_note"] = (
        "Gene symbols from this enriched term were included "
        "in the category gene union."
    )


    if len(context_terms) > 0:

        context_output = context_terms.copy()

        context_output["term_role"] = "context_only"

        context_output["used_for_gene_selection"] = False

        context_output["curation_note"] = (
            "Broad supporting enrichment term; not used "
            "to define heatmap gene membership."
        )


        terms_output = pd.concat(
            [
                terms_output,
                context_output
            ],
            ignore_index=True
        )


    terms_path = os.path.join(
        OUTPUT_DIR,
        slug + "_enrichment_terms_audit.csv"
    )


    terms_output.to_csv(
        terms_path,
        index=False
    )


    # --------------------------------------------------------
    # N. SAVE GENE SELECTION AUDIT
    # --------------------------------------------------------

    gene_to_terms = make_gene_term_map(
        selected_terms
    )


    gene_audit_rows = []


    for gene in sorted(genes_before_curation):

        excluded = gene in exclusions

        gene_audit_rows.append({

            "gene_symbol": gene,

            "enrichment_terms": "; ".join(
                sorted(
                    set(gene_to_terms.get(gene, []))
                )
            ),

            "retained_after_manual_curation": not excluded,

            "manual_exclusion_reason": (
                exclusions.get(gene, "")
            ),

            "found_in_peptide_filtered_annotation": (
                gene in annotated_gene_symbols
            )

        })


    gene_audit = pd.DataFrame(
        gene_audit_rows
    )


    gene_audit_path = os.path.join(
        OUTPUT_DIR,
        slug + "_gene_selection_audit.csv"
    )


    gene_audit.to_csv(
        gene_audit_path,
        index=False
    )


    # --------------------------------------------------------
    # O. SAVE CATEGORY SUMMARY
    # --------------------------------------------------------

    result = {

        "category": category_spec["title"],

        "enrichment_source": category_spec["source"].upper(),

        "enriched_term_rows_used": len(selected_terms),

        "context_only_term_rows": len(context_terms),

        "gene_symbols_before_curation": len(genes_before_curation),

        "gene_symbols_manually_excluded": len(excluded_genes),

        "gene_symbols_after_curation": len(genes_after_curation),

        "gene_symbols_mapped": len(mapped_gene_symbols),

        "unmatched_gene_symbols": len(unmatched_genes),

        "protein_groups_matched": len(matched_ids),

        "protein_groups_passing_coverage": len(coverage_pass_ids),

        "protein_groups_plotted": len(z_scores),

        "PNG_file": png_path,

        "protein_members_file": members_path,

        "enrichment_terms_file": terms_path,

        "gene_audit_file": gene_audit_path

    }


    if pdf_path is not None:

        result["PDF_file"] = pdf_path


    print("\nCATEGORY SUMMARY")
    print("------------------------------")

    for key, value in result.items():

        print(f"{key}: {value}")


    # Save objects for category-overlap analysis.

    category_gene_sets[category_spec["slug"]] = (
        genes_after_curation
    )

    category_protein_sets[category_spec["slug"]] = (
        set(z_scores.index)
    )


    return result


# ------------------------------------------------------------
# 18. GENERATE ALL FIVE HEATMAPS
# ------------------------------------------------------------

category_gene_sets = {}

category_protein_sets = {}

summary_results = []


for category_spec in CATEGORY_SPECS:

    result = create_category_heatmap(
        category_spec
    )

    if result is not None:

        summary_results.append(result)


# ------------------------------------------------------------
# 19. SAVE SUMMARY TABLE
# ------------------------------------------------------------

summary_df = pd.DataFrame(
    summary_results
)


summary_path = os.path.join(
    OUTPUT_DIR,
    "heatmap_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


print("\n")
print("=" * 75)
print("FINAL HEATMAP SUMMARY")
print("=" * 75)


display(summary_df)


# ------------------------------------------------------------
# 20. CALCULATE CATEGORY OVERLAPS
# ------------------------------------------------------------

# Enrichment categories are not perfectly independent.
#
# For example:
# - Mitochondrial ribosomal proteins can belong to both
#   mitochondria and protein synthesis.
# - ECM proteins can also belong to cell-matrix adhesion.
#
# This table makes these overlaps transparent.


overlap_rows = []


category_names = list(category_gene_sets.keys())


for category_a, category_b in combinations(
    category_names,
    2
):

    shared_genes = (
        category_gene_sets[category_a]
        .intersection(category_gene_sets[category_b])
    )


    shared_proteins = (
        category_protein_sets[category_a]
        .intersection(category_protein_sets[category_b])
    )


    overlap_rows.append({

        "category_1": category_a,

        "category_2": category_b,

        "n_shared_gene_symbols": len(shared_genes),

        "shared_gene_symbols": "; ".join(
            sorted(shared_genes)
        ),

        "n_shared_plotted_protein_groups": len(shared_proteins),

        "shared_protein_group_ids": "; ".join(
            sorted(shared_proteins)
        )

    })


overlap_df = pd.DataFrame(
    overlap_rows
)


overlap_path = os.path.join(
    OUTPUT_DIR,
    "category_gene_and_protein_overlaps.csv"
)


overlap_df.to_csv(
    overlap_path,
    index=False
)


print("\nCATEGORY OVERLAPS")
print("------------------------------")

display(overlap_df)


# ------------------------------------------------------------
# 21. CREATE FINAL ZIP PACKAGE
# ------------------------------------------------------------

zip_filename = "PR1_FINAL_HEATMAPS.zip"


with zipfile.ZipFile(
    zip_filename,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for filename in sorted(
        os.listdir(OUTPUT_DIR)
    ):

        full_path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        zip_file.write(
            full_path,
            arcname=filename
        )


print("\n")
print("=" * 75)
print("FINAL HEATMAP PACKAGE CREATED")
print("=" * 75)

print("Output folder:", OUTPUT_DIR)

print("ZIP file:", zip_filename)


# ------------------------------------------------------------
# 22. DOWNLOAD FINAL RESULTS
# ------------------------------------------------------------

from google.colab import files as colab_files

colab_files.download(
    zip_filename
)

INPUT FILES LOADED
------------------------------
Expression matrix: (3834, 76)
Protein annotation: (3834, 5)
Sample annotation: (75, 3)
gProfiler UP: (115, 7)
gProfiler DOWN: (11, 7)

Expression matrix dimensions: (3834, 75)

SAMPLE COUNTS
------------------------------


,plot_group,display_group,n_samples
0,CON_F,CON-F,8
1,CON_M,CON-M,9
2,AS_F,AS-F,21
3,AS_M,AS-M,20



PEPTIDE SUPPORT FILTER
------------------------------
Excluded low-support protein groups: 79
Protein groups retained: 3755


ECM composition and remodeling
Enriched term rows used: 9
Gene symbols before manual curation: 64
Genes excluded by manual curation: ['ACTN1', 'C1QC', 'FGA', 'FGB', 'FGG', 'PLEC', 'PLG', 'SERPINF2']
Gene symbols after curation: 56
Matched protein groups: 61
Unmatched gene symbols: []
Protein groups passing coverage filter: 58
Protein groups included in heatmap: 58

CATEGORY SUMMARY
------------------------------
category: ECM composition and remodeling
enrichment_source: UP
enriched_term_rows_used: 9
context_only_term_rows: 0
gene_symbols_before_curation: 64
gene_symbols_manually_excluded: 8
gene_symbols_after_curation: 56
gene_symbols_mapped: 56
unmatched_gene_symbols: 0
protein_groups_matched: 61
protein_groups_passing_coverage: 58
protein_groups_plotted: 58
PNG_file: PR1_final_heatmaps/01_ECM_heatmap.png
protein_members_file: PR1_final_heatmaps/01_ECM_protei

,category,enrichment_source,enriched_term_rows_used,context_only_term_rows,gene_symbols_before_curation,gene_symbols_manually_excluded,gene_symbols_after_curation,gene_symbols_mapped,unmatched_gene_symbols,protein_groups_matched,protein_groups_passing_coverage,protein_groups_plotted,PNG_file,protein_members_file,enrichment_terms_file,gene_audit_file
0,ECM composition and remodeling,UP,9,0,64,8,56,56,0,61,58,58,PR1_final_heatmaps/01_ECM_heatmap.png,PR1_final_heatmaps/01_ECM_protein_members.csv,PR1_final_heatmaps/01_ECM_enrichment_terms_aud...,PR1_final_heatmaps/01_ECM_gene_selection_audit...
1,Energy metabolism and mitochondria,DOWN,3,0,154,0,154,152,2,153,151,151,PR1_final_heatmaps/02_Mitochondria_heatmap.png,PR1_final_heatmaps/02_Mitochondria_protein_mem...,PR1_final_heatmaps/02_Mitochondria_enrichment_...,PR1_final_heatmaps/02_Mitochondria_gene_select...
2,Protein synthesis and ribosomal machinery,DOWN,7,0,71,0,71,71,0,71,70,70,PR1_final_heatmaps/03_Protein_synthesis_heatma...,PR1_final_heatmaps/03_Protein_synthesis_protei...,PR1_final_heatmaps/03_Protein_synthesis_enrich...,PR1_final_heatmaps/03_Protein_synthesis_gene_s...
3,"Cytoskeleton, adhesion and muscle contraction",UP,5,0,78,16,62,62,0,82,78,78,PR1_final_heatmaps/04_Cytoskeleton_heatmap.png,PR1_final_heatmaps/04_Cytoskeleton_protein_mem...,PR1_final_heatmaps/04_Cytoskeleton_enrichment_...,PR1_final_heatmaps/04_Cytoskeleton_gene_select...
4,"Blood/plasma-associated proteins, complement a...",UP,29,1,71,0,71,71,0,75,74,74,PR1_final_heatmaps/05_Blood_plasma_complement_...,PR1_final_heatmaps/05_Blood_plasma_complement_...,PR1_final_heatmaps/05_Blood_plasma_complement_...,PR1_final_heatmaps/05_Blood_plasma_complement_...



CATEGORY OVERLAPS
------------------------------


,category_1,category_2,n_shared_gene_symbols,shared_gene_symbols,n_shared_plotted_protein_groups,shared_protein_group_ids
0,01_ECM,02_Mitochondria,0,,0,
1,01_ECM,03_Protein_synthesis,0,,0,
2,01_ECM,04_Cytoskeleton,37,ABI3BP; BGN; COL12A1; COL14A1; COL18A1; COL1A1...,39,PG_00016; PG_00046; PG_00214; PG_00257; PG_003...
3,01_ECM,05_Blood_plasma_complement_coagulation,10,A2M; AHSG; APOE; COL1A1; COL1A2; COL3A1; EMILI...,11,PG_00016; PG_00939; PG_00959; PG_00961; PG_009...
4,02_Mitochondria,03_Protein_synthesis,30,C1QBP; CARS2; FARS2; GADD45GIP1; GFM1; LARS2; ...,29,PG_00631; PG_00828; PG_00864; PG_01533; PG_016...
5,02_Mitochondria,04_Cytoskeleton,0,,0,
6,02_Mitochondria,05_Blood_plasma_complement_coagulation,0,,0,
7,03_Protein_synthesis,04_Cytoskeleton,0,,0,
8,03_Protein_synthesis,05_Blood_plasma_complement_coagulation,0,,0,
9,04_Cytoskeleton,05_Blood_plasma_complement_coagulation,10,ACTN1; COL1A1; COL1A2; COL3A1; CSRP1; EMILIN1;...,13,PG_00016; PG_00074; PG_00094; PG_00959; PG_009...




FINAL HEATMAP PACKAGE CREATED
Output folder: PR1_final_heatmaps
ZIP file: PR1_FINAL_HEATMAPS.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>